[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitm-da/da2402/blob/master/data%20collection/practice/data_collection_worksheet.ipynb)

# Practice · data collection

DA2402 · Data Curation and Visualization · Dr. Arun B Ayyar

Ten questions against four files served over HTTP. Each question names a variable. Put your result
in that variable and run the cell. The worked answer sits under **Answer**. Click it open once you
have tried.

**The pages.** A two-page workshop listing, its robots.txt and a saved API response, written for
this worksheet. They live in the course repository and are fetched over the network, so `requests`
does real work and no outside site has to stay up.

- `workshops.html`, `workshops_page2.html`: 18 workshop cards, 12 on the first page
- `workshops_robots.txt`: the crawl rules
- `slots_api.json`: the same 18 workshops as a JSON API response

The full title, the rating, the seat count and the total are each stored somewhere other than the
card's visible text. There is a question on each.


In [ ]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

URL = "https://raw.githubusercontent.com/iitm-da/da2402/master/data%20collection/practice/data/"

print(URL)

### Q1  Fetching the page

Fetch `workshops.html` into `page1` with `requests`. Report the status code and the
`Content-Type` header the server sends back. Note what the server calls the file.

Answer variable `q1`: a tuple `(status_code, content_type)`.

In [ ]:
page1 = ...   # your answer
q1 = ...
q1

<details>
<summary><b>Answer</b></summary>

```python
page1 = requests.get(URL + "workshops.html")
q1 = (page1.status_code, page1.headers["Content-Type"])
q1
```

```
(200, 'text/plain; charset=utf-8')
```

</details>

### Q2  Counting the cards

Parse `page1` with BeautifulSoup and count the workshop cards on it. A card is an
`article` carrying class `workshop`.

Answer variable `q2`: an `int`.

In [ ]:
q2 = ...   # your answer
q2

<details>
<summary><b>Answer</b></summary>

```python
soup = BeautifulSoup(page1.text, "html.parser")
cards = soup.select("article.workshop")
q2 = len(cards)
q2
```

```
12
```

</details>

### Q3  The full title

The first card's link text is cut short with an ellipsis. The full title sits on the link
itself. Report it.

Answer variable `q3`: a `str`.

In [ ]:
q3 = ...   # your answer
q3

<details>
<summary><b>Answer</b></summary>

```python
q3 = cards[0].select_one("h3.title a")["title"]
q3
```

```
'Time series forecasting with statsmodels'
```

</details>

### Q4  Rating from the class name

A card's rating is held as the second class on `span.rating`, written `rating-4`. Pull the
number out for all 12 cards on page 1, in page order.

Answer variable `q4`: a list of 12 ints.

In [ ]:
q4 = ...   # your answer
q4

<details>
<summary><b>Answer</b></summary>

```python
def rating(card):
    return int(card.select_one("span.rating")["class"][1].split("-")[1])

q4 = [rating(c) for c in cards]
q4
```

```
[4, 5, 3, 4, 5, 3, 4, 5, 4, 2, 4, 3]
```

</details>

### Q5  Seat count from the card text

`span.seats` reads `12 seats left`. Total the seats left across page 1.

Answer variable `q5`: an `int`.

In [ ]:
q5 = ...   # your answer
q5

<details>
<summary><b>Answer</b></summary>

```python
q5 = sum(int(re.search(r"(\d+)\s+seats", c.select_one("span.seats").text).group(1))
         for c in cards)
q5
```

```
159
```

</details>

### Q6  TOTAL_WORKSHOPS in a script block

The page declares `TOTAL_WORKSHOPS` inside a `<script>` block, out of reach of a CSS
selector. Read it with a regex over the response text.

Answer variable `q6`: an `int`.

In [ ]:
q6 = ...   # your answer
q6

<details>
<summary><b>Answer</b></summary>

```python
q6 = int(re.search(r"TOTAL_WORKSHOPS\s*=\s*(\d+)", page1.text).group(1))
q6
```

```
18
```

</details>

### Q7  Following the next link

Page 1 links to page 2 through `nav.pager a.next`, and page 2 has no such link. Parse both
pages into one DataFrame with columns `id`, `title`, `mode`, `rating`, `seats_left`, `price`,
`starts_on`, leaving `price` as `None` where a card has none. Report its shape.

Answer variable `q7`: a tuple `(rows, columns)`.

In [ ]:
q7 = ...   # your answer
q7

<details>
<summary><b>Answer</b></summary>

```python
def parse_page(html):
    s = BeautifulSoup(html, "html.parser")
    rows = []
    for c in s.select("article.workshop"):
        price = c.select_one("span.price")
        rows.append({
            "id": c["data-id"],
            "title": c.select_one("h3.title a")["title"],
            "mode": c["data-mode"],
            "rating": int(c.select_one("span.rating")["class"][1].split("-")[1]),
            "seats_left": int(re.search(r"(\d+)", c.select_one("span.seats").text).group(1)),
            "price": price.text.strip() if price else None,
            "starts_on": c.select_one("time")["datetime"],
        })
    nxt = s.select_one("nav.pager a.next")
    return rows, (nxt["href"] if nxt else None)


rows, nxt = parse_page(page1.text)
while nxt:
    more, nxt = parse_page(requests.get(URL + nxt).text)
    rows += more

all_ws = pd.DataFrame(rows)
q7 = all_ws.shape
q7
```

```
(18, 7)
```

</details>

### Q8  Cards with no price

Two of the 18 cards carry no `span.price`. Report their ids, using the frame from Q7.

Answer variable `q8`: a list of 2 strings.

In [ ]:
q8 = ...   # your answer
q8

<details>
<summary><b>Answer</b></summary>

```python
q8 = all_ws.loc[all_ws["price"].isna(), "id"].tolist()
q8
```

```
['W-105', 'W-113']
```

</details>

### Q9  The JSON API

`slots_api.json` holds the same 18 workshops under `results`, with `seats` and `venue`
nested one level down. Flatten the records and total the seats left per mode.

Answer variable `q9`: a Series indexed by mode.

In [ ]:
q9 = ...   # your answer
q9

<details>
<summary><b>Answer</b></summary>

```python
api = requests.get(URL + "slots_api.json").json()
slots = pd.json_normalize(api["results"])

q9 = slots.groupby("mode")["seats.left"].sum()
q9
```

```
mode
in-person    118
online       132
Name: seats.left, dtype: int64
```

</details>

### Q10  What robots.txt permits

Read `workshops_robots.txt` with `urllib.robotparser` and answer four things: may
`my-scraper` fetch `/workshop/W-101`, may it fetch `/register/W-101`, may `GPTBot` fetch
`/workshop/W-101`, and what crawl delay applies to `my-scraper`.

Answer variable `q10`: a tuple of three bools and a number.

In [ ]:
q10 = ...   # your answer
q10

<details>
<summary><b>Answer</b></summary>

```python
from urllib.robotparser import RobotFileParser

rp = RobotFileParser()
rp.parse(requests.get(URL + "workshops_robots.txt").text.splitlines())

q10 = (rp.can_fetch("my-scraper", "/workshop/W-101"),
       rp.can_fetch("my-scraper", "/register/W-101"),
       rp.can_fetch("GPTBot", "/workshop/W-101"),
       rp.crawl_delay("my-scraper"))
q10
```

```
(True, False, False, 5)
```

</details>